# 9.2 不知道時怎麼回答？

# 第 9 章：誠實、安全與適當回應

前置：第7章SFT與第8章分項評估。用能驗證真假的小世界教誠實、修正前提與安全邊界，不把課堂成功推廣成任意開放世界安全保證。外部PKU資料的非商用授權與原始安全標籤要保留。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：盒中球數：看得到纔有根據**

隱藏真值不能當成可見資料；缺資訊要指出缺什麼。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/honesty.svg")))

**先想一想：**盒裡真有3球但題目沒提供，回答3算誠實嗎？

不確定的目標不是每題都說不知道，而是資訊足夠時作答，缺重要資料時指出缺什麼。用有明確隱藏真值的盒子任務容易查證。

**把直覺寫成數學：**訓練標籤由可見資料決定，不由模型看不到的隱藏真值決定。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
examples = [{"visible_count": 3}, {"visible_count": None}]
for e in examples:
    target = str(e["visible_count"]) if e["visible_count"] is not None else "資訊不足，請提供球數。"
    print(e, target)

**如何讀結果：**這是可驗證的標註生成規則；實際LLM需訓練後用未見盒子情境測試。

**只改一件事：**只加一個無關線索，確認它不應變成數量答案。
